# Heart Sound Analysis using Federated Learning and Differential Privacy

This notebook explores federated learning and differential privacy for heart-sound analysis. Make sure dataset paths and dependency versions match your runtime before running.

**Note:** Run cells in order and regenerate evaluation metrics and privacy-accounting outputs; previous execution outputs have been cleared.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/HeartSound_Project"
FEATURE_DIR = os.path.join(
    PROJECT_DIR,
    "fixed_log_mel_all"
)

print("Project directory exists:",
      os.path.exists(PROJECT_DIR))

print("Feature directory exists:",
      os.path.exists(FEATURE_DIR))

feature_files = [
    f for f in os.listdir(FEATURE_DIR)
    if f.endswith(".npy")
]

print("Feature files:", len(feature_files))
print("First 5:", feature_files[:5])

In [ ]:
!pip install -q opacus

In [ ]:
import torch
import opacus

print("PyTorch:", torch.__version__)
print("Opacus:", opacus.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import shutil

SOURCE_DIR = "/content/drive/MyDrive/HeartSound_Project/fixed_log_mel_all"
LOCAL_FEATURE_DIR = "/content/fixed_log_mel_all"

if not os.path.exists(LOCAL_FEATURE_DIR):
    print("Copying features to local Colab storage...")
    shutil.copytree(SOURCE_DIR, LOCAL_FEATURE_DIR)
    print("Copy complete!")
else:
    print("Local features already exist.")

feature_files = [
    f for f in os.listdir(LOCAL_FEATURE_DIR)
    if f.endswith(".npy")
]

print("Local feature files:", len(feature_files))

In [ ]:
import pandas as pd
import os
import glob

PROJECT_DIR = "/content/drive/MyDrive/HeartSound_Project"
REFERENCE_DIRS = [
    "training-a",
    "training-b",
    "training-c",
    "training-d",
    "training-e",
    "training-f"
]

reference_parts = []

for folder in REFERENCE_DIRS:

    ref_path = os.path.join(
        PROJECT_DIR,
        folder,
        "REFERENCE.csv"
    )

    df = pd.read_csv(
        ref_path,
        header=None,
        names=["recording", "label"]
    )

    df["folder"] = folder

    reference_parts.append(df)


reference = pd.concat(
    reference_parts,
    ignore_index=True
)

print("Total recordings:", len(reference))
print("\nFirst 5:")
print(reference.head())

print("\nClass distribution:")
print(reference["label"].value_counts())

In [ ]:
dataset_df = reference.copy()

dataset_df["target"] = dataset_df["label"].map({
    -1: 0,
    1: 1
})

dataset_df["feature_path"] = dataset_df["recording"].apply(
    lambda x: os.path.join(
        LOCAL_FEATURE_DIR,
        x + ".npy"
    )
)

dataset_df["feature_exists"] = dataset_df["feature_path"].apply(
    os.path.exists
)

print("Total:", len(dataset_df))
print("Features found:", dataset_df["feature_exists"].sum())

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Recreate the original 2268-sample training set
train_df, temp_df = train_test_split(
    dataset_df,
    test_size=0.30,
    stratify=dataset_df["target"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["target"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
NUM_CLIENTS = 3

client_source = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

client_dfs = list(
    np.array_split(client_source, NUM_CLIENTS)
)

print("Number of clients:", len(client_dfs))

for i, client_df in enumerate(client_dfs):
    print(
        f"Client {i+1}: {len(client_df)} samples | "
        f"Classes: {client_df['target'].value_counts().sort_index().to_dict()}"
    )

print(
    "Total client samples:",
    sum(len(df) for df in client_dfs)
)

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader


class HeartSoundDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        feature = np.load(row["feature_path"]).astype(np.float32)

        # (128, 954) → (1, 128, 954)
        feature = torch.from_numpy(feature).unsqueeze(0)

        target = torch.tensor(
            row["target"],
            dtype=torch.long
        )

        return feature, target

In [ ]:
BATCH_SIZE = 16

client_loaders = []

for i, client_df in enumerate(client_dfs):

    client_dataset = HeartSoundDataset(client_df)

    client_loader = DataLoader(
        client_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    client_loaders.append(client_loader)

    print(
        f"Client {i+1}: "
        f"{len(client_dataset)} samples, "
        f"{len(client_loader)} batches"
    )

In [ ]:
features, targets = next(iter(client_loaders[0]))

print("Client 1 feature shape:", features.shape)
print("Client 1 target shape:", targets.shape)
print("Targets:", targets)

In [ ]:
import torch
import torch.nn as nn


class FLHeartSoundCRNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm = nn.LSTM(
            input_size=1024,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        x = self.cnn(x)

        batch_size = x.size(0)

        x = x.permute(0, 3, 1, 2)

        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )

        x, _ = self.lstm(x)

        x = x.mean(dim=1)

        return self.classifier(x)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

NUM_ROUNDS = 10
LOCAL_EPOCHS = 1
LEARNING_RATE = 0.001

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Clients:", NUM_CLIENTS)
print("Federated rounds:", NUM_ROUNDS)
print("Local epochs:", LOCAL_EPOCHS)

In [ ]:
def federated_average(client_states, client_sizes):

    total_samples = sum(client_sizes)

    global_state = {}

    for key in client_states[0].keys():

        global_state[key] = sum(
            client_states[i][key].float()
            * (client_sizes[i] / total_samples)
            for i in range(len(client_states))
        )

    return global_state

In [ ]:
def train_local_model(
    model,
    loader,
    epochs=1,
    lr=0.001
):

    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    for epoch in range(epochs):

        running_loss = 0.0
        correct = 0
        total = 0

        for features, targets in loader:

            features = features.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            outputs = model(features)

            loss = criterion(
                outputs,
                targets
            )

            loss.backward()
            optimizer.step()

            running_loss += (
                loss.item() *
                features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == targets
            ).sum().item()

            total += targets.size(0)

    return (
        running_loss / total,
        correct / total
    )

In [ ]:
# ============================================
# ONE FEDERATED LEARNING ROUND
# ============================================

global_model = FLHeartSoundCRNN().to(device)

global_state = global_model.state_dict()

client_states = []
client_sizes = []

for client_id, client_loader in enumerate(client_loaders):

    print(f"\nTraining Client {client_id + 1}...")

    # Create local model with the current global weights
    local_model = FLHeartSoundCRNN().to(device)

    local_model.load_state_dict(global_state)

    # Train locally
    loss, accuracy = train_local_model(
        local_model,
        client_loader,
        epochs=LOCAL_EPOCHS,
        lr=LEARNING_RATE
    )

    print(
        f"Client {client_id + 1} "
        f"Loss: {loss:.4f} "
        f"Accuracy: {accuracy:.4f}"
    )

    # Store local model parameters
    client_states.append(
        {
            key: value.detach().cpu().clone()
            for key, value in local_model.state_dict().items()
        }
    )

    client_sizes.append(
        len(client_loader.dataset)
    )

# FedAvg
new_global_state = federated_average(
    client_states,
    client_sizes
)

global_model.load_state_dict(
    new_global_state
)

print("\n================================")
print("Federated Round 1 Complete")
print("================================")
print("Global model successfully aggregated.")

In [ ]:
NUM_ROUNDS = 10
LOCAL_EPOCHS = 1

fl_history = []

global_model = FLHeartSoundCRNN().to(device)

for round_num in range(1, NUM_ROUNDS + 1):

    print(f"\n{'='*50}")
    print(f"FEDERATED ROUND {round_num}/{NUM_ROUNDS}")
    print(f"{'='*50}")

    global_state = global_model.state_dict()

    client_states = []
    client_sizes = []
    client_accuracies = []

    # -------------------------
    # LOCAL CLIENT TRAINING
    # -------------------------
    for client_id, client_loader in enumerate(client_loaders):

        local_model = FLHeartSoundCRNN().to(device)

        local_model.load_state_dict(global_state)

        loss, accuracy = train_local_model(
            local_model,
            client_loader,
            epochs=LOCAL_EPOCHS,
            lr=LEARNING_RATE
        )

        print(
            f"Client {client_id + 1}: "
            f"Loss={loss:.4f}, "
            f"Accuracy={accuracy:.4f}"
        )

        client_states.append({
            key: value.detach().cpu().clone()
            for key, value in local_model.state_dict().items()
        })

        client_sizes.append(
            len(client_loader.dataset)
        )

        client_accuracies.append(accuracy)

    # -------------------------
    # FEDAVG
    # -------------------------
    new_global_state = federated_average(
        client_states,
        client_sizes
    )

    global_model.load_state_dict(
        new_global_state
    )

    avg_client_accuracy = sum(
        client_accuracies
    ) / len(client_accuracies)

    fl_history.append({
        "round": round_num,
        "avg_client_accuracy": avg_client_accuracy
    })

    print(
        f"\nRound {round_num} complete."
    )

    print(
        f"Average client accuracy: "
        f"{avg_client_accuracy:.4f}"
    )

print("\n" + "="*50)
print("FEDERATED LEARNING COMPLETE")
print("="*50)

In [ ]:
# Create test dataset and DataLoader
test_dataset = HeartSoundDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Test samples:", len(test_dataset))
print("Test batches:", len(test_loader))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

global_model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = global_model(features)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            targets.cpu().numpy()
        )

accuracy = accuracy_score(
    all_targets,
    all_predictions
)

precision = precision_score(
    all_targets,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_targets,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_targets,
    all_predictions,
    zero_division=0
)

cm = confusion_matrix(
    all_targets,
    all_predictions
)

TN, FP, FN, TP = cm.ravel()

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

print("Federated Learning Test Results")
print("================================")
print(f"Accuracy:      {accuracy:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1-score:      {f1:.4f}")
print(f"Sensitivity:   {sensitivity:.4f}")
print(f"Specificity:   {specificity:.4f}")

print("\nConfusion Matrix:")
print(cm)

In [ ]:
import torch
import opacus

print("PyTorch:", torch.__version__)
print("Opacus:", opacus.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from opacus import PrivacyEngine

In [ ]:
def train_local_model_dp(
    model,
    loader,
    epochs=1,
    lr=0.001,
    max_grad_norm=1.0,
    noise_multiplier=1.0
):

    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    privacy_engine = PrivacyEngine()

    model, optimizer, loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=max_grad_norm
    )

    running_loss = 0.0
    correct = 0
    total = 0

    for epoch in range(epochs):

        for features, targets in loader:

            features = features.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            outputs = model(features)

            loss = criterion(
                outputs,
                targets
            )

            loss.backward()
            optimizer.step()

            running_loss += (
                loss.item() *
                features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == targets
            ).sum().item()

            total += targets.size(0)

    epsilon = privacy_engine.get_epsilon(
        delta=1e-5
    )

    return (
        running_loss / total,
        correct / total,
        epsilon
    )

In [ ]:
class DPHeartSoundCRNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm = nn.LSTM(
            input_size=1024,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        x = self.cnn(x)

        batch_size = x.size(0)

        x = x.permute(0, 3, 1, 2)

        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )

        x, _ = self.lstm(x)

        x = x.mean(dim=1)

        return self.classifier(x)


print(DPHeartSoundCRNN())

In [ ]:
from opacus.layers import DPLSTM

In [ ]:
class DPHeartSoundCRNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm = DPLSTM(
            input_size=1024,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        x = self.cnn(x)

        batch_size = x.size(0)

        x = x.permute(0, 3, 1, 2)

        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )

        x, _ = self.lstm(x)

        x = x.mean(dim=1)

        return self.classifier(x)


dp_test_model = DPHeartSoundCRNN().to(device)

print(dp_test_model)

In [ ]:
dp_loss, dp_accuracy, dp_epsilon = train_local_model_dp(
    model=dp_test_model,
    loader=client_loaders[0],
    epochs=1,
    lr=0.001,
    max_grad_norm=1.0,
    noise_multiplier=1.0
)

print("\n================================")
print("DP Client 1 Test")
print("================================")
print(f"Loss:       {dp_loss:.4f}")
print(f"Accuracy:   {dp_accuracy:.4f}")
print(f"Epsilon:    {dp_epsilon:.4f}")
print("Delta:      1e-5")

In [ ]:
# ============================================
# ONE FEDERATED LEARNING + DIFFERENTIAL
# PRIVACY ROUND
# ============================================

dp_global_model = DPHeartSoundCRNN().to(device)

dp_global_state = dp_global_model.state_dict()

dp_client_states = []
dp_client_sizes = []
dp_client_accuracies = []
dp_client_epsilons = []

for client_id, client_loader in enumerate(client_loaders):

    print(f"\nTraining DP Client {client_id + 1}...")

    # Create local DP model
    local_dp_model = DPHeartSoundCRNN().to(device)

    # Start from current global model
    local_dp_model.load_state_dict(dp_global_state)

    # DP local training
    loss, accuracy, epsilon = train_local_model_dp(
        model=local_dp_model,
        loader=client_loader,
        epochs=1,
        lr=0.001,
        max_grad_norm=1.0,
        noise_multiplier=1.0
    )

    print(
        f"Client {client_id + 1}: "
        f"Loss={loss:.4f}, "
        f"Accuracy={accuracy:.4f}, "
        f"Epsilon={epsilon:.4f}"
    )

    # Save local model parameters
    dp_client_states.append({
        key: value.detach().cpu().clone()
        for key, value in local_dp_model.state_dict().items()
    })

    dp_client_sizes.append(
        len(client_loader.dataset)
    )

    dp_client_accuracies.append(accuracy)
    dp_client_epsilons.append(epsilon)


# ============================================
# FEDERATED AVERAGING
# ============================================

new_dp_global_state = federated_average(
    dp_client_states,
    dp_client_sizes
)

dp_global_model.load_state_dict(
    new_dp_global_state
)


print("\n" + "=" * 50)
print("FL + DP ROUND 1 COMPLETE")
print("=" * 50)

print(
    "Average client accuracy:",
    f"{sum(dp_client_accuracies) / len(dp_client_accuracies):.4f}"
)

print(
    "Client epsilons:",
    [round(e, 4) for e in dp_client_epsilons]
)

In [ ]:
# ============================================
# FL + DP: ROUNDS 2–5
# ============================================

DP_START_ROUND = 2
DP_END_ROUND = 5

dp_history = []

for round_num in range(DP_START_ROUND, DP_END_ROUND + 1):

    print(f"\n{'='*50}")
    print(f"FL + DP ROUND {round_num}/{DP_END_ROUND}")
    print(f"{'='*50}")

    dp_global_state = dp_global_model.state_dict()

    dp_client_states = []
    dp_client_sizes = []
    dp_client_accuracies = []
    dp_client_epsilons = []

    # -------------------------
    # LOCAL DP TRAINING
    # -------------------------
    for client_id, client_loader in enumerate(client_loaders):

        local_dp_model = DPHeartSoundCRNN().to(device)

        local_dp_model.load_state_dict(
            dp_global_state
        )

        loss, accuracy, epsilon = train_local_model_dp(
            model=local_dp_model,
            loader=client_loader,
            epochs=1,
            lr=0.001,
            max_grad_norm=1.0,
            noise_multiplier=1.0
        )

        print(
            f"Client {client_id + 1}: "
            f"Loss={loss:.4f}, "
            f"Accuracy={accuracy:.4f}, "
            f"Epsilon={epsilon:.4f}"
        )

        dp_client_states.append({
            key: value.detach().cpu().clone()
            for key, value in local_dp_model.state_dict().items()
        })

        dp_client_sizes.append(
            len(client_loader.dataset)
        )

        dp_client_accuracies.append(accuracy)
        dp_client_epsilons.append(epsilon)

    # -------------------------
    # FEDAVG
    # -------------------------
    new_dp_global_state = federated_average(
        dp_client_states,
        dp_client_sizes
    )

    dp_global_model.load_state_dict(
        new_dp_global_state
    )

    avg_accuracy = sum(
        dp_client_accuracies
    ) / len(dp_client_accuracies)

    max_epsilon = max(
        dp_client_epsilons
    )

    dp_history.append({
        "round": round_num,
        "average_client_accuracy": avg_accuracy,
        "max_epsilon": max_epsilon
    })

    print(
        f"\nRound {round_num} complete."
    )

    print(
        f"Average client accuracy: "
        f"{avg_accuracy:.4f}"
    )

    print(
        f"Maximum client epsilon: "
        f"{max_epsilon:.4f}"
    )

print("\n" + "="*50)
print("FL + DP ROUNDS 2–5 COMPLETE")
print("="*50)

print("\nDP history:")
for result in dp_history:
    print(result)

In [ ]:
# ============================================================
# HEART SOUND FL + DP — RECOVERY / SETUP CELL
# ============================================================

# -----------------------------
# 1. Imports
# -----------------------------
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from opacus import PrivacyEngine
from opacus.layers import DPLSTM


# -----------------------------
# 2. Device
# -----------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# -----------------------------
# 3. Project paths
# -----------------------------
PROJECT_DIR = "/content/drive/MyDrive/HeartSound_Project"

FEATURE_DIR = os.path.join(
    PROJECT_DIR,
    "fixed_log_mel_all"
)

print("Feature directory exists:",
      os.path.exists(FEATURE_DIR))


# -----------------------------
# 4. Load metadata
# -----------------------------
REFERENCE_DIRS = [
    "training-a",
    "training-b",
    "training-c",
    "training-d",
    "training-e",
    "training-f"
]

reference_parts = []

for folder in REFERENCE_DIRS:

    ref_path = os.path.join(
        PROJECT_DIR,
        folder,
        "REFERENCE.csv"
    )

    df = pd.read_csv(
        ref_path,
        header=None,
        names=["recording", "label"]
    )

    df["folder"] = folder

    reference_parts.append(df)


reference = pd.concat(
    reference_parts,
    ignore_index=True
)


# -----------------------------
# 5. Create dataset dataframe
# -----------------------------
dataset_df = reference.copy()

dataset_df["target"] = dataset_df["label"].map({
    -1: 0,
     1: 1
})

dataset_df["feature_path"] = dataset_df["recording"].apply(
    lambda x: os.path.join(
        FEATURE_DIR,
        x + ".npy"
    )
)

dataset_df["feature_exists"] = (
    dataset_df["feature_path"].apply(os.path.exists)
)

print("\nTotal recordings:", len(dataset_df))
print(
    "Features found:",
    dataset_df["feature_exists"].sum()
)
print(
    "Features missing:",
    (~dataset_df["feature_exists"]).sum()
)


# -----------------------------
# 6. SAME train/val/test split
# -----------------------------
train_df, temp_df = train_test_split(
    dataset_df,
    test_size=0.30,
    stratify=dataset_df["target"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["target"],
    random_state=42
)

print("\nDataset split:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


# -----------------------------
# 7. Dataset class
# -----------------------------
class HeartSoundDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        feature = np.load(
            row["feature_path"]
        ).astype(np.float32)

        feature = torch.from_numpy(
            feature
        ).unsqueeze(0)

        target = torch.tensor(
            row["target"],
            dtype=torch.long
        )

        return feature, target


# -----------------------------
# 8. Create 3 FL clients
# -----------------------------
NUM_CLIENTS = 3

client_source = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

client_dfs = list(
    np.array_split(
        client_source,
        NUM_CLIENTS
    )
)

print("\nFederated clients:")

for i, client_df in enumerate(client_dfs):

    print(
        f"Client {i+1}:",
        len(client_df),
        "samples |",
        client_df["target"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

print(
    "Total client samples:",
    sum(len(x) for x in client_dfs)
)


# -----------------------------
# 9. Client DataLoaders
# -----------------------------
BATCH_SIZE = 16

client_loaders = []

for client_df in client_dfs:

    dataset = HeartSoundDataset(
        client_df
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    client_loaders.append(loader)


# -----------------------------
# 10. Test DataLoader
# -----------------------------
test_dataset = HeartSoundDataset(
    test_df
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("\nTest samples:", len(test_dataset))


# ============================================================
# 11. Plain FL CRNN
# ============================================================
class FLHeartSoundCRNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(
                1, 16,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm = nn.LSTM(
            input_size=1024,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        x = self.cnn(x)

        batch_size = x.size(0)

        x = x.permute(
            0, 3, 1, 2
        )

        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )

        x, _ = self.lstm(x)

        x = x.mean(dim=1)

        return self.classifier(x)


# ============================================================
# 12. DP-compatible CRNN
# ============================================================
class DPHeartSoundCRNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(
                1, 16,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm = DPLSTM(
            input_size=1024,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        x = self.cnn(x)

        batch_size = x.size(0)

        x = x.permute(
            0, 3, 1, 2
        )

        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )

        x, _ = self.lstm(x)

        x = x.mean(dim=1)

        return self.classifier(x)


# ============================================================
# 13. FedAvg
# ============================================================
def federated_average(
    client_states,
    client_sizes
):

    total_samples = sum(
        client_sizes
    )

    global_state = {}

    for key in client_states[0].keys():

        global_state[key] = sum(
            client_states[i][key].float()
            *
            (
                client_sizes[i]
                / total_samples
            )
            for i in range(
                len(client_states)
            )
        )

    return global_state


# ============================================================
# 14. DP local training
# ============================================================
def train_local_model_dp(
    model,
    loader,
    epochs=1,
    lr=0.001,
    max_grad_norm=1.0,
    noise_multiplier=1.0
):

    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    privacy_engine = PrivacyEngine()

    model, optimizer, loader = (
        privacy_engine.make_private(
            module=model,
            optimizer=optimizer,
            data_loader=loader,
            noise_multiplier=noise_multiplier,
            max_grad_norm=max_grad_norm
        )
    )

    running_loss = 0.0
    correct = 0
    total = 0

    for epoch in range(epochs):

        for features, targets in loader:

            features = features.to(
                device,
                non_blocking=True
            )

            targets = targets.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(features)

            loss = criterion(
                outputs,
                targets
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item()
                *
                features.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            correct += (
                predictions == targets
            ).sum().item()

            total += targets.size(0)

    epsilon = privacy_engine.get_epsilon(
        delta=1e-5
    )

    return (
        running_loss / total,
        correct / total,
        epsilon
    )


# ============================================================
# 15. DP configuration
# ============================================================
DP_NOISE_MULTIPLIER = 1.0
DP_MAX_GRAD_NORM = 1.0
DP_DELTA = 1e-5
DP_LOCAL_EPOCHS = 1
DP_LEARNING_RATE = 0.001

print("\n==========================================")
print("FL + DP RECOVERY SETUP COMPLETE")
print("==========================================")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Clients:", NUM_CLIENTS)
print("DP noise multiplier:", DP_NOISE_MULTIPLIER)
print("DP max grad norm:", DP_MAX_GRAD_NORM)
print("DP delta:", DP_DELTA)